In [1]:
import os
import pandas as pd
import numpy as np

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Paths
weekly_path = '../data/processed/weekly_sku_demand.csv'
weekly_df = pd.read_csv(weekly_path)
weekly_df['Week_Start'] = pd.to_datetime(weekly_df['Week_Start'])

print(f"Loaded weekly demand history for {weekly_df['StockCode'].nunique()} qualified SKUs.")

Loaded weekly demand history for 1676 qualified SKUs.


In [2]:
# Compute SKU-level demand statistics over active history
inventory_summary = weekly_df.groupby('StockCode').agg(
    Avg_Weekly_Demand=('Quantity', 'mean'),
    Demand_Std_Dev=('Quantity', 'std'),
    Recent_Forecast_Demand=('Quantity', 'tail_4_mean') if 'tail_4_mean' in weekly_df else ('Quantity', lambda x: x.tail(4).mean())
).reset_index()

# Round intermediate statistics for clarity
inventory_summary['Avg_Weekly_Demand'] = inventory_summary['Avg_Weekly_Demand'].round(2)
inventory_summary['Demand_Std_Dev'] = inventory_summary['Demand_Std_Dev'].round(2)
inventory_summary['Recent_Forecast_Demand'] = inventory_summary['Recent_Forecast_Demand'].round(2)

inventory_summary.head()

,StockCode,Avg_Weekly_Demand,Demand_Std_Dev,Recent_Forecast_Demand
0,10002,109.53,155.29,52.75
1,10120,15.46,20.59,13.75
2,10125,30.04,35.10,22.00
3,10133,51.58,65.22,163.75
4,10135,44.84,89.95,34.50


In [3]:
# Fixed Analytical Assumptions (Configurable parameters)
LEAD_TIME_WEEKS = 2        # Assumed 2-week supplier replenishment lead time
Z_SERVICE_LEVEL = 1.65     # 95% Target Cycle Service Level

# 1. Lead Time Demand (LTD) = Avg Demand * Lead Time
inventory_summary['Lead_Time_Demand'] = (inventory_summary['Avg_Weekly_Demand'] * LEAD_TIME_WEEKS).round(2)

# 2. Safety Stock (SS) = Z * std_dev * sqrt(Lead Time)
inventory_summary['Safety_Stock'] = (
    Z_SERVICE_LEVEL * inventory_summary['Demand_Std_Dev'] * np.sqrt(LEAD_TIME_WEEKS)
).round(0)

# 3. Reorder Point (ROP) = Lead Time Demand + Safety Stock
inventory_summary['Reorder_Point'] = (
    inventory_summary['Lead_Time_Demand'] + inventory_summary['Safety_Stock']
).round(0)

# 4. Transparent Inventory Simulation (Initialized at 1.25 * ROP with uniform variation)
# Explicitly labeled SIMULATED to maintain full data integrity
np.random.seed(42)  # For exact repeatability
variation_factor = np.random.uniform(0.5, 1.8, size=len(inventory_summary))

inventory_summary['Simulated_Current_Stock'] = (
    inventory_summary['Reorder_Point'] * variation_factor
).round(0)

# Inventory Position (Current Stock under static pipeline assumption)
inventory_summary['Simulated_Inventory_Position'] = inventory_summary['Simulated_Current_Stock']

inventory_summary.head()

,StockCode,Avg_Weekly_Demand,Demand_Std_Dev,Recent_Forecast_Demand,Lead_Time_Demand,Safety_Stock,Reorder_Point,Simulated_Current_Stock,Simulated_Inventory_Position
0,10002,109.53,155.29,52.75,219.06,362.00,581.00,573.00,573.00
1,10120,15.46,20.59,13.75,30.92,48.00,79.00,137.00,137.00
2,10125,30.04,35.10,22.00,60.08,82.00,142.00,206.00,206.00
3,10133,51.58,65.22,163.75,103.16,152.00,255.00,326.00,326.00
4,10135,44.84,89.95,34.50,89.68,210.00,300.00,211.00,211.00


In [4]:
# Calculate days/weeks of supply on hand
inventory_summary['Weeks_of_Supply'] = (
    inventory_summary['Simulated_Current_Stock'] / inventory_summary['Avg_Weekly_Demand'].replace(0, 1)
).round(1)

# Risk Threshold Logic
def evaluate_stockout_risk(row):
    if row['Simulated_Inventory_Position'] <= row['Reorder_Point']:
        return 'HIGH'
    elif row['Simulated_Inventory_Position'] <= (row['Reorder_Point'] + row['Safety_Stock']):
        return 'MEDIUM'
    else:
        return 'LOW'

def evaluate_overstock_risk(row):
    # Overstocked if current stock exceeds 3x Reorder Point or >12 weeks of supply
    if row['Simulated_Inventory_Position'] > (row['Reorder_Point'] * 2.5) or row['Weeks_of_Supply'] > 12:
        return 'HIGH'
    elif row['Simulated_Inventory_Position'] > (row['Reorder_Point'] * 1.8):
        return 'MEDIUM'
    else:
        return 'LOW'

inventory_summary['Stockout_Risk'] = inventory_summary.apply(evaluate_stockout_risk, axis=1)
inventory_summary['Overstock_Risk'] = inventory_summary.apply(evaluate_overstock_risk, axis=1)

print("--- INVENTORY RISK DISTRIBUTION ---")
print("Stockout Risk Levels:")
print(inventory_summary['Stockout_Risk'].value_counts())
print("\nOverstock Risk Levels:")
print(inventory_summary['Overstock_Risk'].value_counts())

--- INVENTORY RISK DISTRIBUTION ---
Stockout Risk Levels:
Stockout_Risk
MEDIUM    735
HIGH      653
LOW       288
Name: count, dtype: int64

Overstock Risk Levels:
Overstock_Risk
LOW     1666
HIGH      10
Name: count, dtype: int64


In [ ]:
# Save complete inventory table to data/processed
output_path = '../data/processed/inventory_analysis_table.csv'
inventory_summary.to_csv(output_path, index=False)
print(f"Saved inventory analysis table to {output_path} successfully.")